# Token Prediction Analysis

Analyzes model predictions at each position: confidence, surprisal, difficulty profiling, prediction agreement across layers, and rank trajectories.

**References:**
- Geva et al. (2022) "Transformer Feed-Forward Layers Build Predictions by Promoting Concepts"

## Why This Matters

Token prediction analysis examines model confidence, surprisal, difficulty, and agreement across positions. Position-level prediction quality metrics reveal which parts of the input are easy vs. hard for the model, and how prediction difficulty correlates with internal computation.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_prediction_analysis import (
    per_token_confidence,
    surprisal_profile,
    token_difficulty_profile,
    prediction_agreement_by_layer,
    rank_trajectory,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
print(f"Model: {cfg.n_layers} layers, {cfg.d_vocab} vocab")

In [ ]:
# 1. Per-token confidence
conf = per_token_confidence(model, tokens)
print(f"Mean confidence: {conf['mean_confidence']:.4f}")
print(f"Most confident position: {conf['most_confident_position']} (p={conf['top1_probs'][conf['most_confident_position']]:.4f})")
print(f"Least confident position: {conf['least_confident_position']} (p={conf['top1_probs'][conf['least_confident_position']]:.4f})")
print(f"\nPer-position:")
for i in range(len(tokens)):
    print(f"  pos {i}: top1={conf['top1_tokens'][i]:3d}, prob={conf['top1_probs'][i]:.4f}, entropy={conf['entropies'][i]:.3f}")

In [ ]:
# 2. Surprisal profile
surp = surprisal_profile(model, tokens)
print(f"Mean surprisal: {surp['mean_surprisal']:.4f} nats")
print(f"Accuracy: {surp['accuracy']:.1%}")
print(f"Most surprised at position: {surp['max_surprisal_position']} (surprisal={surp['surprisals'][surp['max_surprisal_position']]:.4f})")
print(f"\nPer-position surprisal:")
for i in range(len(surp['surprisals'])):
    mark = 'correct' if surp['correct_predictions'][i] else 'wrong'
    print(f"  pos {i}->{i+1}: surprisal={surp['surprisals'][i]:.4f} ({mark})")

In [ ]:
# 3. Token difficulty profile
diff = token_difficulty_profile(model, tokens, n_samples=3)
print(f"Hardest position: {diff['hardest_position']}")
print(f"Easiest position: {diff['easiest_position']}")
print(f"\nRelative difficulty per position:")
for i in range(len(tokens)):
    bar = '#' * int(diff['relative_difficulty'][i] * 20)
    print(f"  pos {i}: {diff['relative_difficulty'][i]:.3f} {bar}")

In [ ]:
# 4. Prediction agreement by layer
agree = prediction_agreement_by_layer(model, tokens, pos=-1, top_k=5)
print(f"First agreement layer: {agree['first_agreement_layer']}")
print(f"Consensus fraction: {agree['consensus_fraction']:.2f}")
print(f"\nLayer-by-layer top-5 overlap with final:")
for l in range(model.cfg.n_layers):
    print(f"  Layer {l}: overlap={agree['agreement_with_final'][l]:.2f}, top-5={agree['layer_top_k'][l+1].tolist()}")
print(f"  Final: top-5={agree['layer_top_k'][-1].tolist()}")

In [ ]:
# 5. Rank trajectory
targets = [0, 10, 25, 50]
rt = rank_trajectory(model, tokens, target_tokens=targets, pos=-1)
print("Token rank trajectories through layers:")
for t in targets:
    ranks = rt['ranks'][t]
    print(f"  Token {t:3d}: ranks={ranks.tolist()}, final_rank={rt['final_ranks'][t]}")